# Opponent modeling: does opponent history help us forecast the next action?

The opponent-forecasting workflow in the shared project. We preserve five simulator families, action priors, Naive Bayes, matched logistic models, card-equity controls, multiple history windows, and a Bayesian style filter behind one shared contract.

**Target:** the opponent's next observed legal action in two-player Leduc. PokerBench and the existing Hero decision checkpoints remain a separate task; these labels do not teach the best response to an opponent. Predictive gains do not establish human tilt, a GTO policy, or improved poker payoff.

The reusable implementation is in `poker_decision.opponents`. See [integration and source attribution](../docs/meeting4-integration.md) and [experiment guide](../docs/opponent-experiments.md). Original submitted results are kept separately from new canonical runs.


## How forecasting fits the project

Our existing PokerBench model learns **which action the reference policy selects for Hero**. This notebook adds a different supervised task: **which action will this opponent take next?** Chronological simulated play supplies the observed-action labels, so we can test opponent modeling without inventing a ground-truth label for “tilt.”

| Contribution | What it adds to the canonical workflow |
| --- | --- |
| Session generators and recording contracts | Repeat opponents across hands, preserve visibility, separate labels and audit metadata, and reproduce/replay data |
| Simple probabilistic baselines | Establish what constant action rates and Naive Bayes can predict before adding more complex models |
| Opportunity-based summaries and logistic models | Compare the same current situation with and without prior behavior; preserve counts and uncertainty from sparse history |
| History-window experiments | Test short/long memory and show where old observations become misleading after a switch |
| Visible-card equity controls | Test a reduced hand-strength representation while keeping current opportunity cues matched |
| Bayesian style estimator | Update an inferred aggression/folding distribution as actions are observed and plot the belief over time |
| Scientific evaluation and reconciliation | Group repeated identities, freeze choices before test scoring, report all variants and preserve source-specific verification limits |

The submissions implement different versions of these pieces; the [contribution map](../docs/meeting4-integration.md) identifies each source. This notebook uses their retained ideas through one package, one input/target contract and matched evaluation.

**Connection to our decision model:** the separate [Hold’em integration](holdem_opponent_integration.ipynb) now reuses these causal summaries as extra transformer context on real histories. It predicts observed human actions; learned forecast probabilities are not injected. The [22-fit follow-up](../docs/meeting5-ablation-results.md) finds conditional, seed-dependent benefits. Learning better responses still requires appropriate response targets or repeated-play reward evaluation. A learned sequential history encoder remains future work. See the [results index](../docs/results/README.md) for the distinct studies.


## 1. Setup

Local: run `uv sync --locked --extra analysis --extra experiments`, launch Jupyter from this repository, and select its kernel. The bootstrap below can clone/install the repository in Colab CPU. It does not download PokerBench, train a transformer, or use an API key.


In [ ]:
import runpy
import subprocess
import sys
from pathlib import Path

EXTRAS = "analysis,experiments"

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/poker_decision").is_dir()), None
)
if ROOT is None and Path("/content").is_dir():
    ROOT = Path("/content/Poker-Decision-Model")
    if not ROOT.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/nmokey/Poker-Decision-Model.git", str(ROOT)],
            check=True,
        )
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project checkout.")
sys.path.insert(0, str(ROOT / "src"))
setup = runpy.run_path(
    str(ROOT / "scripts/notebook_setup.py"), init_globals={"ROOT": ROOT, "EXTRAS": EXTRAS}
)
IN_COLAB = setup["IN_COLAB"]

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from poker_decision.opponents.data import Config, FeatureSpec, feature_matrix, history_summary
from poker_decision.opponents.experiment import (
    final_evaluation,
    prepare,
    read_jsonl,
    select,
    verify_manifest,
)

print("Repository:", ROOT)

## 2. Choose a fixed protocol before generating data

Default: five card-dependent behavior profiles, 10 independent opponents per profile, two sessions per opponent, 60 hands per session. Opponent instances are split 60/20/20 within profile and switch direction; repeated sessions stay together. History resets at every session. Held-out opponents come from known families.

`policy_family` options:
- `weighted`: tight/passive/aggressive/balanced log weights plus two-way switches; rank and pairs influence play.
- `categorical`: card-independent passive/aggressive policies and switches, retaining the simple modular prototype.
- `table`: card-dependent tight/aggressive probability tables and switches.
- `residual`: five aggression/fold styles plus switches, inspired by the Bayesian submission.
- `cfr`: an approximate CFR+ reference, tight/loose distortions, and switches in both directions.

`observer_policy="cfr"` uses the same approximate solver as the fixed observer in every session; otherwise the observer samples uniformly. Teacher exploitability measures the teacher alone. `observer_card=False` makes the predictor public-only; with `True`, only the observer's own card is added.

Changing configuration, implementation, or recorded software versions creates a new run directory. This canonical protocol differs from the original submissions; do not compare its scores directly with their historical results.


In [ ]:
CFG = Config(
    seed=20261106,
    policy_family="weighted",
    opponents_per_profile=10,
    sessions_per_opponent=2,
    hands_per_session=60,
    observer_policy="uniform",
    observer_card=True,
)
OUTPUT = ROOT / "artifacts/opponents"
TIMING = "completed_hands"  # alternative: "all_prior", including earlier actions of this hand
RUN_FINAL = False  # set True after reviewing and freezing the validation choices
DATA = prepare(CFG, OUTPUT)
MANIFEST = verify_manifest(DATA)
print("Dataset:", DATA)
print("Counts:", MANIFEST["counts"])
print("Engine replay:", MANIFEST["audit"])
print("Versions:", MANIFEST["protocol"]["versions"])

## 3. Inspect the input/target boundary and chronology

`inputs` contains public board, round, pot/contributions, the current-hand public action prefix, legal actions, and optionally the observer's own card. The observed action is a separate `target_index`. Session/opponent IDs and ordering are outer metadata; profiles, switches, seeds, hidden cards, and full replay actions live in audit artifacts.

`fold`, `check_call`, `bet_raise` are engine action categories; local target indices refer to the supplied legal menu. Check/call and opening bet/raise share engine IDs. No illegal move is remapped.


In [ ]:
train = read_jsonl(DATA / "train.jsonl")
example = train[0]
print("Visible input:", json.dumps(example["inputs"], indent=2))
print(
    "Separate target:",
    example["target_index"],
    example["inputs"]["candidates"][example["target_index"]],
)
first_session = [r for r in train if r["session_id"] == example["session_id"]]
display(
    pd.DataFrame(
        [
            dict(
                hand=r["hand_index"],
                decision=r["decision_index"],
                round=r["inputs"]["public"]["round"],
                menu=r["inputs"]["candidates"],
                target=r["target_index"],
            )
            for r in first_session
        ]
    )
)

## 4. Inspect causal, opportunity-based histories

The summary is taken before the target is observed. In the default `completed_hands` mode, current-hand opponent actions enter summaries only at the next hand. The no-history model still sees the entire public prefix within the current hand.

Windows 4/16/64 take the last W eligible decisions, then match round and exact legal menu. The menu model also retains the alternative “match menu, then take the latest 10 comparable opportunities.” Counts, denominators, and +1-smoothed rates make sparse and empty histories explicit. The strength controls include current round/menu cues on both sides to avoid crediting history with a new context cue.


In [ ]:
empty = history_summary([], example["inputs"], 16)
print("Empty history:", empty)
print("Public feature matrix:", feature_matrix(train[:10], FeatureSpec()).shape)
print("With history:", feature_matrix(train[:10], FeatureSpec(window=16, timing=TIMING)).shape)

## 5. Fit on training rows; choose settings on validation sessions

This step reads train/validation labels only. Set `selection_method="grouped_cv"` to choose C using five folds of whole training opponents, retaining the linked Colab's CV idea while keeping repeated identities together. History-window and Bayesian choices still use validation. It selects C for every logistic model, a summary window for the declared primary history comparison, and temperature/hazard for the Bayesian filter. All proposed history windows are retained in the final report.

- `prior` and `uniform`: legal-normalized simple controls.
- `no_history` and `history_4/16/64/all`: the same standardized logistic architecture, optimizing cross-entropy over legal actions during training.
- `candidate_no_history` / `candidate_history`: the linked Colab's binary candidate-scoring objective followed by legal-menu softmax, retained as a distinct control.
- `menu_no_history` / `menu_history`: separate logistic regressions per legal menu. Sparse or unseen menus fall back to a smoothed training prior.
- `naive_bayes_*`: simple smoothed, discretized feature-token baselines with train-only vocabulary.
- `strength*`: exact showdown equity against uniformly random unknown cards, with matched opportunity controls. Equity is not Villain's true strength or a betting-conditioned range.
- `bayesian`: 25 aggression/fold hypotheses updated after feedback, without simulator labels or switch times. Its feedback timing matches the selected history protocol; its per-hand hazard accounts for skipped hands.

Fitted numeric models and frozen choices are saved before any final-test inference. Optimizer failures stop the run.


In [ ]:
SELECTION = select(DATA, timing=TIMING, selection_method="validation")  # or "grouped_cv"
choices = json.loads((SELECTION / "selection.json").read_text())
display(pd.DataFrame(choices["candidates"]))
print("Validation-selected summary:", choices["selected_history"])
print("Validation-selected Bayesian settings:", choices["bayesian"])
print("Saved selection:", SELECTION)

## 6. Explicit held-out evaluation

After choices are fixed, set `RUN_FINAL=True` and run this cell. Earlier observed test actions are available to later predictions according to `TIMING`; model weights are never fitted on test labels. Report accuracy, tie-expected accuracy, log loss, Brier score, counts, history/switch slices, and paired whole-opponent bootstrap intervals that include both repeated sessions. Small slices are descriptive; intervals do not capture retraining or simulator-design uncertainty.

The persisted result is reused after hash checks. A changed source/runtime, corrupted input, or different evaluation settings is rejected rather than silently replacing a run.


In [ ]:
RESULTS = None
if RUN_FINAL:
    RESULTS = final_evaluation(DATA, SELECTION)
    display(pd.DataFrame(RESULTS["overall"]).T)
    print(RESULTS["interpretation"])
    print(json.dumps(RESULTS["comparisons"]["history_selected"], indent=2))
else:
    print("Validation complete. Review choices before enabling RUN_FINAL.")

## 7. Inspect behavior changes, examples, and belief trajectories

Longer history can remain useful overall while becoming stale immediately after a switch. Compare the predeclared models on the same decisions; do not choose the best test window and present it as an independent winner. Bins by accumulated observations are descriptive stages of sessions, not a causal test of collecting more history.


In [ ]:
if RESULTS is not None:
    diagnostics = pd.DataFrame(RESULTS["diagnostics"])
    display(diagnostics[diagnostics["model"].isin(["no_history", "history_selected", "bayesian"])])
    for item in RESULTS["examples"]:
        print(json.dumps(item, indent=2))
    overall = pd.DataFrame(RESULTS["overall"]).T
    overall.loc[
        ["no_history", "history_4", "history_16", "history_64", "bayesian"],
        ["log_loss", "accuracy"],
    ].plot.bar(subplots=True, figsize=(10, 6), legend=False)
    plt.tight_layout()
    plt.show()
    predictions = read_jsonl(SELECTION / "test_predictions.jsonl")
    sessions = json.loads((DATA / "sessions.json").read_text())
    sid = next(
        s["session_id"] for s in sessions if s["split"] == "test" and s["switch_hand"] is not None
    )
    trace = pd.DataFrame(
        [
            dict(
                hand=r["hand_index"],
                aggression=r["bayesian_style_mean"][0],
                folding=r["bayesian_style_mean"][1],
            )
            for r in predictions
            if r["session_id"] == sid
        ]
    )
    ax = trace.plot(
        x="hand", y=["aggression", "folding"], title="Inferred style in one switching test session"
    )
    switch = next(s["switch_hand"] for s in sessions if s["session_id"] == sid)
    ax.axvline(switch, color="black", linestyle="--", label="True switch (audit only)")
    ax.legend()
    plt.show()

## 8. Completion conditions and next experiment

- **Data done:** deterministic session generation, one readable input and separate target, disjoint opponent identities, and all recorded hands replay successfully.
- **Forecasting done:** no-history and summary models evaluated on the same held-out decisions, with negative results retained and saved choices/provenance.
- **Variety retained:** priors, Naive Bayes, per-menu logistic, equity controls, 4/16/64/all windows, Bayesian feedback and five generator families share reusable code.
- **Review done:** inspect five predictions, all reported window scores and switch slices; distinguish submitted historical results from this run.

A learned sequential encoder remains future work. Connecting predictions to Hero's existing decision model still needs opponent-dependent decision targets or repeated-play reward evaluation. The present workflow does not alter PokerBench labels or the deployed demo.

The summary-to-transformer connection is implemented separately in the
[Hold’em workflow](holdem_opponent_integration.ipynb), with observed-action targets.
The [recorded Leduc results](../docs/meeting4-results.md) remain this notebook's
own evaluation, distinct from that later integration.


## Reuse the project's candidate scorer

`opponent_task` converts this saved split into the same `DecisionExample` interface
used in [decision modeling](decision_model.ipynb). It reuses the causal features
above, keeps whole-opponent groups outside inputs, and declares a different question.
A future neural forecast can call `encode_examples(task.examples, tokenizer,
max_length=..., question=task.question)` and `train_epoch` with these targets.
Keep the saved opponent splits; use the opponent metrics for abstract action categories.
This adapter does not train a neural forecast or demonstrate better Hero decisions.


In [ ]:
from poker_decision.tasks import opponent_task

forecast_task = opponent_task(train, spec=FeatureSpec(window=16, timing=TIMING))
print(forecast_task.question, "|", forecast_task.label_semantics)
print("Shared input:", forecast_task.examples[0].inputs)
print("External split group:", forecast_task.groups[0])